<a href="https://colab.research.google.com/github/justorfc/Est_Python_R_2026_1/blob/main/8_Semana_08_Sesion_01_Bondad_Ajuste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Llegamos a un momento verdaderamente crítico para cualquier futuro ingeniero: el momento en que deben dejar de "adivinar" mirando una gráfica y comenzar a tomar decisiones respaldadas por rigor estadístico. La bondad de ajuste es el puente que valida todo lo que han aprendido en las semanas anteriores.

Se ha incorporado la premisa: *"No aceptamos el modelo, solo evaluamos la evidencia en contra"*.

Aquí está la propuesta:

---

### 📘 Estructura del Notebook: Semana 8 - Sesión 1

**Título del Notebook:** `Semana_08_Sesion_01_Bondad_Ajuste.ipynb`

#### **Celda 1: Markdown (Encabezado y Fase 1: IA)**

# SEMANA 8: Pruebas de Bondad de Ajuste
**Asignatura:** Estadística Aplicada con Python y R
**Programas:** Ingeniería Agrícola, Agroindustrial y Civil

## Primera Hora: Actividad "Estudia y Aprende"
En las semanas anteriores aprendimos a observar la forma de los datos y a proponer modelos teóricos (Normal, Lognormal, Weibull, etc.). Pero, ¿cómo demostramos matemáticamente que el modelo propuesto realmente "encaja" con la realidad? Antes de usar código, interactúa con tu IA para dominar la lógica de las pruebas de hipótesis.

**PROMPT 1 - INICIO:**
> Actúa como tutor experto en pruebas de bondad de ajuste aplicadas a ingeniería.
> Tema: Pruebas KS, Anderson-Darling y Chi-cuadrado.
> 1. Explica qué significa "bondad de ajuste".
> 2. Explica qué es hipótesis nula y alternativa en este contexto.
> 3. Explica qué es valor-p y cómo se interpreta.
> 4. Explica la prueba Kolmogorov-Smirnov.
> 5. Explica Anderson-Darling y cómo difiere de KS.
> 6. Explica Chi-cuadrado para datos discretos.
> 7. Da ejemplos aplicados a ingeniería.
> Hazme 3 preguntas para verificar comprensión y corrige mis respuestas.

#### **Celda 2: Markdown (Fase 2: Orientación Docente - Contexto y Lógica)**

## Segunda Hora: Orientación Docente en Python
Ajustar un modelo significa medir la "distancia" entre lo que observamos en la realidad y lo que el modelo teórico predice. Para hacerlo formalmente, usamos el contraste de hipótesis:

* **Hipótesis Nula ($H_0$):** Los datos provienen de la distribución propuesta (El modelo ES un buen ajuste).
* **Hipótesis Alternativa ($H_1$):** Los datos NO provienen de esa distribución.

**La Regla de Oro del Valor-p:** El valor-p es la probabilidad de observar nuestros datos si $H_0$ fuera cierta. Si el valor-p es muy pequeño (usualmente $< 0.05$), tenemos evidencia para **rechazar** el modelo.
*Nota crucial en ingeniería:* Nunca "aceptamos" un modelo como la verdad absoluta; simplemente decimos que *no hay evidencia suficiente para rechazarlo*.

#### **Celda 3: Código Python (Visualización del Concepto de Ajuste)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
np.random.seed(42)

# Simulamos la resistencia a la tensión (MPa) de 100 cables de acero.
# Físicamente, la resistencia suele seguir una distribución Normal o Weibull.
# Aquí generaremos datos estrictamente Normales.
resistencia_cables = stats.norm.rvs(loc=250, scale=15, size=100)

# ==========================================
# 1. VISUALIZAR vs PROBAR FORMALMENTE
# ==========================================
plt.figure(figsize=(10, 5))
# Histograma de los datos observados
sns.histplot(resistencia_cables, bins=15, stat='density', color='steelblue', alpha=0.6, label='Datos Observados')

# Superponemos la curva Normal Teórica (usando la media y std de los datos)
xmin, xmax = plt.xlim()
x_continuo = np.linspace(xmin, xmax, 100)
pdf_teorica = stats.norm.pdf(x_continuo, loc=np.mean(resistencia_cables), scale=np.std(resistencia_cables))

plt.plot(x_continuo, pdf_teorica, 'r-', lw=2, label='Modelo Normal Teórico')
plt.title('Inspección Visual del Ajuste (Resistencia a la Tensión)')
plt.xlabel('Resistencia (MPa)')
plt.ylabel('Densidad')
plt.legend()
plt.show()

# Interpretación: Visualmente, parece un buen ajuste. Pero el ojo humano engaña.
# Necesitamos una prueba matemática para respaldar la decisión de usar este modelo
# en el diseño de un puente (Ing. Civil) o un tractor (Ing. Agrícola).

#### **Celda 4: Código Python (Introducción a Kolmogorov-Smirnov y Anderson-Darling)**

In [ ]:
# ==========================================
# 2. LAS PRUEBAS FORMALES (Introducción)
# ==========================================
# Vamos a estandarizar los datos (Z-score) para la prueba KS estándar
resistencia_estandarizada = (resistencia_cables - np.mean(resistencia_cables)) / np.std(resistencia_cables)

print("--- 1. PRUEBA KOLMOGOROV-SMIRNOV (KS) ---")
# Comparamos contra una Normal estándar ('norm')
ks_stat, ks_pvalor = stats.kstest(resistencia_estandarizada, 'norm')
print(f"Estadístico KS: {ks_stat:.4f}")
print(f"Valor-p: {ks_pvalor:.4f}")

# Interpretación KS: Como el Valor-p (aprox 0.94) es MAYOR a 0.05,
# NO TENEMOS EVIDENCIA para rechazar la Hipótesis Nula.
# Conclusión inicial: Es razonable asumir que la resistencia es Normal.

print("\n--- 2. PRUEBA ANDERSON-DARLING (AD) ---")
# AD es más rigurosa, da mucho peso a los valores extremos (las colas),
# lo cual es vital en ingeniería (las fallas ocurren en los extremos).
ad_result = stats.anderson(resistencia_cables, dist='norm')
print(f"Estadístico AD: {ad_result.statistic:.4f}")
print("Valores Críticos según nivel de significancia:")
for i in range(len(ad_result.critical_values)):
    sig_level = ad_result.significance_level[i]
    crit_val = ad_result.critical_values[i]
    print(f"  Para {sig_level}% el valor crítico es {crit_val:.3f}")

# Interpretación AD: En Anderson-Darling no miramos un valor-p directo,
# sino que comparamos el estadístico con el valor crítico.
# Como nuestro Estadístico (0.246) es MENOR que el valor crítico al 5% (0.759),
# NO rechazamos la hipótesis nula. El modelo Normal sobrevive a la prueba estricta.

#### **Celda 5: Markdown (Transición a R y RMarkdown)**

```markdown
## Trabajo Autónomo: Transición a R y RMarkdown

En R, la aplicación de estas pruebas es extremadamente directa gracias a funciones nativas y paquetes especializados.

**Paso 1: Validación en Colab (Entorno R)**
1. Crea un nuevo Notebook en Colab, asegúrate de cambiar el entorno a **R**.
2. Ejecuta el siguiente código. Nota el uso de la librería `nortest` para la prueba de Anderson-Darling.

```R
# Instalar paquete si no está disponible
# install.packages("nortest")
library(nortest)

# Generar los mismos datos simulados de resistencia
set.seed(42)
resistencia_r <- rnorm(100, mean=250, sd=15)

print("--- Prueba de Kolmogorov-Smirnov (KS) ---")
# En R, ks.test compara directamente con los parámetros estimados
ks_test_result <- ks.test(resistencia_r, "pnorm", mean=mean(resistencia_r), sd=sd(resistencia_r))
print(ks_test_result)

print("--- Prueba de Anderson-Darling (AD) ---")
ad_test_result <- ad.test(resistencia_r)
print(ad_test_result)

```

**Paso 2: Documentación en Posit Cloud y Despliegue en RPubs**

1. Abre tu espacio en **Posit Cloud** y crea un nuevo documento **RMarkdown**.
2. Transfiere el código de R y ejecútalo.
3. Lo más importante: Escribe un párrafo técnico explicando, con tus propias palabras, qué significa que el *p-value* en la prueba de Anderson-Darling en R sea mayor a 0.05. ¿Por qué esto nos da "luz verde" para usar el modelo Normal en cálculos posteriores?
4. Compila (Knit) a HTML y publica en **RPubs**.
